In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


nltk.download('punkt')
nltk.download('punkt_tab')   
nltk.download('stopwords')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\saksh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\saksh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\saksh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
#loading dataset
#RAW DATA
df = pd.read_csv("twitter.zip")
df


,clean_text,category
0,when modi promised “minimum government maximum...,-1.0
1,talk all the nonsense and continue all the dra...,0.0
2,what did just say vote for modi welcome bjp t...,1.0
3,asking his supporters prefix chowkidar their n...,1.0
4,answer who among these the most powerful world...,1.0
...,...,...
162975,why these 456 crores paid neerav modi not reco...,-1.0
162976,dear rss terrorist payal gawar what about modi...,-1.0
162977,did you cover her interaction forum where she ...,0.0
162978,there big project came into india modi dream p...,0.0


In [3]:
#sample text
df.head()

,clean_text,category
0,when modi promised “minimum government maximum...,-1.0
1,talk all the nonsense and continue all the dra...,0.0
2,what did just say vote for modi welcome bjp t...,1.0
3,asking his supporters prefix chowkidar their n...,1.0
4,answer who among these the most powerful world...,1.0


In [4]:
#class distribution
df['category'].value_counts()

category
 1.0    72250
 0.0    55213
-1.0    35510
Name: count, dtype: int64

In [5]:

#Exploing no of samples
df.info
len(df)

162980

In [6]:
#DATA PREPROCESSING
df.isnull().sum()
df = df.dropna(subset=['category'])

In [7]:

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    # 1. Lowercasing
    text = text.lower()
    
    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # 3. Remove special characters and punctuation
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 4. Tokenization
    tokens = word_tokenize(text)
    
    # 5. Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    
    # 6. Stemming
    tokens = [stemmer.stem(word) for word in tokens]
    
    return " ".join(tokens)
 

df['clean_text'] = df['clean_text'].fillna("").astype(str)
df['processed_text'] = df['clean_text'].apply(preprocess_text)


C:\Users\saksh\AppData\Local\Temp\ipykernel_3000\1975245248.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['clean_text'] = df['clean_text'].fillna("").astype(str)
C:\Users\saksh\AppData\Local\Temp\ipykernel_3000\1975245248.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['processed_text'] = df['clean_text'].apply(preprocess_text)


In [16]:
#feature Engneering
tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=5000)

X = tfidf.fit_transform(df['processed_text'])
y = df['category']

print("Shape:", X.shape)

Shape: (162973, 5000)


In [17]:
# MODEL TRAINING
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [18]:

#Regression
# Create model
lr = LogisticRegression(max_iter=1000)

# Train model (learning happens here)
lr.fit(X_train, y_train)

# Predict
y_pred_lr = lr.predict(X_test)

#Naive Bayes
# Create model
nb = MultinomialNB()

# Train model
nb.fit(X_train, y_train)

# Predict
y_pred_nb = nb.predict(X_test)

#Decison tree
# Create model
dt = DecisionTreeClassifier()

# Train model
dt.fit(X_train, y_train)

# Predict
y_pred_dt = dt.predict(X_test)



In [19]:
#EVALUATION

def evaluate_model(name, y_test, y_pred):
    print(f"----- {name} -----")
    
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall   :", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score :", f1_score(y_test, y_pred, average='weighted'))
    
    print("\n")

# Evaluate all models
evaluate_model("Logistic Regression", y_test, y_pred_lr)
evaluate_model("Naive Bayes", y_test, y_pred_nb)
evaluate_model("Decision Tree", y_test, y_pred_dt)

----- Logistic Regression -----
Accuracy : 0.841448074858107
Precision: 0.8429039413659435
Recall   : 0.841448074858107
F1 Score : 0.8403834281195338


----- Naive Bayes -----
Accuracy : 0.7076852277956742
Precision: 0.7268684991544709
Recall   : 0.7076852277956742
F1 Score : 0.697820053422935


----- Decision Tree -----
Accuracy : 0.7646878355576009
Precision: 0.7638255527497756
Recall   : 0.7646878355576009
F1 Score : 0.7641838929760706




In [26]:
#Example
def predict_sentiment(text):
    processed = preprocess_text(text)
    vector = tfidf.transform([processed])
    prediction = lr.predict(vector)[0]

    if prediction == 1:
        return "Positive "
    elif prediction == -1:
        return "Negative "
    else:
        return "Neutral "
print(predict_sentiment("Very disappointing experience"))
print(predict_sentiment("I am happy with this"))
print(predict_sentiment(""))

Negative 
Positive 
